# Dependencies

In [ ]:
import tensorflow as tf
import kagglehub
import keras
import matplotlib.pyplot as plt

from google.colab import drive
drive.mount('/content/drive')

# Get to coding

## Initial setup
We start by defining our dimensions and batch sizes for inceptionV3, in order for it to work properly according to its documentation.

In [ ]:
# Defining image dimensions and batch size
IMG_SIZE = (299, 299) # InceptionV3 expects inputs of at least 75x75, but 299x299 is standard for its pre-trained weights.
BATCH_SIZE = 32 #

DATA_DIR_TRAIN = '/content/drive/MyDrive/grocery_dataset/train'
print("Training data is located at ", DATA_DIR_TRAIN)

DATA_DIR_VAL = '/content/drive/MyDrive/grocery_dataset/val'
print("Validation data is located at ", DATA_DIR_VAL)

VAL_SPLIT = 0.33

EPOCHS = 20

## Splitting the data

### Datasplit for training

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR_TRAIN,
    labels='inferred',
    label_mode='int',
    validation_split=VAL_SPLIT,
    subset='training',
    seed=1337,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

### Datasplit for validation

In [ ]:
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR_VAL,
    labels='inferred',
    label_mode='int',
    validation_split=VAL_SPLIT,
    subset='validation',
    seed=1337,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

### Class naming

In [ ]:
class_names = train_ds.class_names
num_classes = len(class_names)
print(f"Total number of classes: {num_classes}")
print(f"Product categories: {class_names}")

## Optimizing for performance
Adding caching is a good idea, since we dont want the GPU or CPU to wait for data loading.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

## Setting up the model
### Loading the pretrained base of InceptionV3
Using InceptionV3'd ImageNet pre-trained dataset, but excluding the classification layer.

In [ ]:
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras import layers, models

# Instantiate the base model
base_model = InceptionV3(
    input_shape=IMG_SIZE + (3,), # Expected input shape
    include_top=False,           # Excluding the 1000-class classifier layer
    weights='imagenet'           # Use weights pre-trained on ImageNet
)

### Freezing the base model
We freeze the weights of the base of InceptionV3, preventing the random training on our dataset from destroying the base knowledge and patterns learned from ImageNet.

In [ ]:
base_model.trainable = False # FREEZE the layers

### Adding our own little network
We add our own little network (a new head) on top of the frozen base. This head is responsible for taking the features extracted by InceptionV3  and classifying them into our product categories (num_classes).

In [ ]:
global_average_layer = layers.GlobalAveragePooling2D() # Reducing the 3D feature map to a 1D vector
prediction_layer = layers.Dense(num_classes, activation='softmax') # Our final classification layer

data_augmentation = models.Sequential([
  layers.RandomFlip("horizontal_and_vertical"),   # Flips images randomly
  layers.RandomRotation(0.2),                     # Rotates up to 20% (72 degrees)
  layers.RandomZoom(0.2),                         # Zooms in/out by 20%
])

# Combine the base and the head using the Functional API
model = models.Sequential([
    # Add the augmentation
    data_augmentation,

    # Add a preprocessing layer for InceptionV3's required input scaling (if not handled by the data loader)
    layers.Lambda(tf.keras.applications.inception_v3.preprocess_input, input_shape=(299, 299, 3)),
    base_model,
    global_average_layer,
    layers.Dropout(0.2), # Good practice to prevent overfitting
    prediction_layer
])

# Print the summary to see your new, smaller model head
model.summary()

model.save("model.keras")

Compiling with adam optimizer to up the performance a little. Adam combines the best features of two optimizers i.e Momentum and RMSprop.


In [ ]:
from tensorflow.keras.optimizers import Adam

optimizer = Adam(learning_rate=0.0001)
model.compile(optimizer=optimizer, loss=keras.losses.SparseCategoricalCrossentropy(), metrics=['accuracy'])

## Time to train!
Training without fine tuning first...

In [ ]:
history = model.fit(

    train_ds,
    epochs=EPOCHS,
    validation_data=val_ds,

)

Epoch 1/20
38/38 ━━━━━━━━━━━━━━━━━━━━ 614s 14s/step - accuracy: 0.0376 - loss: 3.9069 - val_accuracy: 0.1706 - val_loss: 3.4915
Epoch 2/20
38/38 ━━━━━━━━━━━━━━━━━━━━ 460s 12s/step - accuracy: 0.1579 - loss: 3.4595 - val_accuracy: 0.2082 - val_loss: 3.2144
Epoch 3/20
38/38 ━━━━━━━━━━━━━━━━━━━━ 458s 12s/step - accuracy: 0.1920 - loss: 3.1564 - val_accuracy: 0.2696 - val_loss: 2.9682
Epoch 4/20
38/38 ━━━━━━━━━━━━━━━━━━━━ 502s 12s/step - accuracy: 0.2552 - loss: 2.9692 - val_accuracy: 0.3242 - val_loss: 2.7449
Epoch 5/20
38/38 ━━━━━━━━━━━━━━━━━━━━ 457s 12s/step - accuracy: 0.3229 - loss: 2.7066 - val_accuracy: 0.4044 - val_loss: 2.5450
Epoch 6/20
 6/38 ━━━━━━━━━━━━━━━━━━━━ 3:47 7s/step - accuracy: 0.3943 - loss: 2.5815

In [ ]:
# Extract data from the history object
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

# Plot the training and validation accuracy
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

# Plot the training and validation loss
plt.subplot(1, 2, 2)
plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.show()

print("Training accuracy: ", acc)
print("Validation accuracy: ", val_acc)

## Stress testing the model

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.inception_v3 import preprocess_input

def TestAccuracy(path):
  img_path = path

  # Loading the image and resizing it to 299x299 (InceptionV3 standard)
  img = image.load_img(img_path, target_size=(299, 299))

  # Converting image to a numpy array and adding a "batch" dimension
  # (Keras expects a batch of images, even if it's just one)
  img_array = image.img_to_array(img)
  img_array = np.expand_dims(img_array, axis=0)

  img_array = preprocess_input(img_array)

  # Running the prediction
  predictions = model.predict(img_array)

  # Getting the class with the highest probability
  # score = tf.nn.softmax(predictions[0])
  score = predictions[0]
  predicted_class = class_names[np.argmax(score)]
  confidence = np.max(predictions[0]) * 100

  print(f"I am {confidence:.2f}% sure this is a(n) {predicted_class}.")

TestAccuracy('/content/drive/MyDrive/grocery_dataset/test/tomato/cherry/613f07bb-bf5e-447d-96e5-997e16499566.jpg')
TestAccuracy('/content/drive/MyDrive/grocery_dataset/test/onion/shallots/d537fd87-6ebe-43f3-a514-820b0c5b969e.jpg')


